### Machine Learning Models - SVM, LR, Random Forest

In [ ]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score
from sentence_transformers import SentenceTransformer
from sklearn.preprocessing import StandardScaler
from scipy.stats import spearmanr
import warnings
warnings.filterwarnings('ignore')

labeled_df = cleaned_df[cleaned_df['label'].notna()]
labeled_df['label'] = labeled_df['label'].apply(lambda x: 1 if x == 'Hallucination' else 0)


if 'p(Hallucination)' not in labeled_df.columns:
    raise ValueError("Missing `p(Hallucination)` field in dataset. Ensure probability values exist.")

labeled_df['combined_input'] = (
    labeled_df['hyp'].fillna('') + ' ' +
    labeled_df['src'].fillna('') + ' ' +
    labeled_df['tgt'].fillna('') + ' ' +
    labeled_df['ref'].fillna('') + ' ' +
    labeled_df['model'].fillna('') + ' ' +
    labeled_df['task'].fillna('')
)

val_df = labeled_df[labeled_df['Dataset'].str.contains('Val')]
test_df = labeled_df[labeled_df['Dataset'].str.contains('Test')]
print("Val-Test split size: ", val_df.shape, test_df.shape)

X_val = val_df['combined_input']
y_val = val_df['label']
p_val = val_df['p(Hallucination)']

X_test = test_df['combined_input']
y_test = test_df['label']
p_test = test_df['p(Hallucination)']
sbert_model = SentenceTransformer('all-MiniLM-L6-v2')

def get_glove_embeddings(texts, glove_path='/root/.cache/kagglehub/datasets/watts2/glove6b50dtxt/versions/1/glove.6B.50d.txt'):
    glove_dict = {}
    with open(glove_path, 'r') as file:
        for line in file:
            values = line.split()
            word = values[0]
            vector = np.asarray(values[1:], dtype='float32')
            glove_dict[word] = vector
    def embed(text):
        tokens = text.split()
        vectors = [glove_dict.get(t, np.zeros(50)) for t in tokens]
        return np.mean(vectors, axis=0) if vectors else np.zeros(50)
    return np.array([embed(text) for text in texts])

vectorizer = TfidfVectorizer()
X_val_tfidf = vectorizer.fit_transform(X_val)
X_test_tfidf = vectorizer.transform(X_test)
X_val_sbert = sbert_model.encode(X_val.tolist(), convert_to_tensor=False)
X_test_sbert = sbert_model.encode(X_test.tolist(), convert_to_tensor=False)
X_val_glove = get_glove_embeddings(X_val)
X_test_glove = get_glove_embeddings(X_test)

embedding_methods = {
    'TF-IDF': (X_val_tfidf, X_test_tfidf),
    'Sentence-BERT': (X_val_sbert, X_test_sbert),
    'GloVe': (X_val_glove, X_test_glove),
}

models = {
    'Logistic Regression': LogisticRegression(max_iter=1000),
    'SVM': SVC(kernel='linear', probability=True),
    'Random Forest': RandomForestClassifier(n_estimators=100),
}

def evaluate_model(model, X_val, X_test, y_val, y_test, p_test):
    model.fit(X_val, y_val)
    y_pred = model.predict(X_test)

    # Get predicted probabilities for Spearman correlation
    if hasattr(model, "predict_proba"):
        y_pred_probs = model.predict_proba(X_test)[:, 1]  # Probability of hallucination
    else:
        y_pred_probs = model.decision_function(X_test)  # For SVM, decision function serves as confidence score

    # Compute Accuracy
    accuracy = accuracy_score(y_test, y_pred)

    # Compute Spearman Correlation
    spearman_corr, _ = spearmanr(p_test, y_pred_probs)

    print(f"Accuracy: {accuracy:.2f}")
    print(f"Spearman Correlation: {spearman_corr:.4f}")
    print("\nClassification Report:")
    print(classification_report(y_test, y_pred))

    return accuracy, spearman_corr

results = []

for embed_name, (X_val_emb, X_test_emb) in embedding_methods.items():
    for model_name, model in models.items():
        print(f"Training {model_name} with {embed_name} embeddings...")
        accuracy, spearman_corr = evaluate_model(model, X_val_emb, X_test_emb, y_val, y_test, p_test)
        results.append({'Model': model_name, 'Embedding': embed_name, 'Accuracy': accuracy, 'Spearman Correlation': spearman_corr})
        print("=" * 80)


results_df = pd.DataFrame(results)
print("\nFinal Comparison:")
print(results_df)

Val-Test split size:  (1000, 12) (2995, 12)
Training Logistic Regression with TF-IDF embeddings...
Accuracy: 0.61
Spearman Correlation: 0.1756

Classification Report:
              precision    recall  f1-score   support

           0       0.64      0.86      0.73      1834
           1       0.51      0.23      0.32      1161

    accuracy                           0.61      2995
   macro avg       0.57      0.54      0.52      2995
weighted avg       0.59      0.61      0.57      2995

Training SVM with TF-IDF embeddings...
Accuracy: 0.60
Spearman Correlation: 0.1586

Classification Report:
              precision    recall  f1-score   support

           0       0.65      0.74      0.69      1834
           1       0.48      0.38      0.42      1161

    accuracy                           0.60      2995
   macro avg       0.56      0.56      0.56      2995
weighted avg       0.58      0.60      0.59      2995

Training Random Forest with TF-IDF embeddings...
Accuracy: 0.62
Spearman

### Deep Learning Models - LSTM, Bi-LSTM, CNN

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import accuracy_score, classification_report
from sentence_transformers import SentenceTransformer
from scipy.stats import spearmanr
from tqdm import tqdm

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

labeled_df = cleaned_df[cleaned_df['label'].notna()]
labeled_df['label'] = labeled_df['label'].apply(lambda x: 1 if x == 'Hallucination' else 0)

if 'p(Hallucination)' not in labeled_df.columns:
    raise ValueError("Missing `p(Hallucination)` field in dataset. Ensure probability values exist.")

labeled_df['combined_input'] = (
    labeled_df['hyp'].fillna('') + ' ' +
    labeled_df['src'].fillna('') + ' ' +
    labeled_df['tgt'].fillna('') + ' ' +
    labeled_df['ref'].fillna('') + ' ' +
    labeled_df['model'].fillna('') + ' ' +
    labeled_df['task'].fillna('')
)

val_df = labeled_df[labeled_df['Dataset'].str.contains('Val')]
test_df = labeled_df[labeled_df['Dataset'].str.contains('Test')]

X_val = val_df['combined_input'].tolist()
y_val = np.array(val_df['label'].tolist(), dtype=np.float32)
p_val = np.array(val_df['p(Hallucination)'].tolist(), dtype=np.float32)

X_test = test_df['combined_input'].tolist()
y_test = np.array(test_df['label'].tolist(), dtype=np.float32)
p_test = np.array(test_df['p(Hallucination)'].tolist(), dtype=np.float32)

# TF-IDF Embeddings
def get_tfidf_embeddings(texts):
    vectorizer = TfidfVectorizer(max_features=5000)
    return vectorizer.fit_transform(texts).toarray()

# GloVe Embeddings
def get_glove_embeddings(texts, glove_path='/root/.cache/kagglehub/datasets/watts2/glove6b50dtxt/versions/1/glove.6B.50d.txt'):
    glove_dict = {}
    with open(glove_path, 'r', encoding='utf-8') as file:
        for line in file:
            values = line.split()
            word = values[0]
            vector = np.asarray(values[1:], dtype='float32')
            glove_dict[word] = vector

    def embed(text):
        tokens = text.split()
        vectors = [glove_dict.get(t, np.zeros(50)) for t in tokens]
        return np.mean(vectors, axis=0) if vectors else np.zeros(50)

    return np.array([embed(text) for text in texts])

# SBERT Embeddings
sbert_model = SentenceTransformer('all-MiniLM-L6-v2')

# Generate embeddings
X_val_tfidf = get_tfidf_embeddings(X_val)
X_test_tfidf = get_tfidf_embeddings(X_test)

X_val_glove = get_glove_embeddings(X_val)
X_test_glove = get_glove_embeddings(X_test)

X_val_sbert = np.array(sbert_model.encode(X_val, convert_to_numpy=True), dtype=np.float32)
X_test_sbert = np.array(sbert_model.encode(X_test, convert_to_numpy=True), dtype=np.float32)

# Define Deep Learning Models
class BaseDLModel(nn.Module):
    def __init__(self, input_dim, hidden_dim=128):
        super(BaseDLModel, self).__init__()
        self.hidden_dim = hidden_dim

class RNNClassifier(BaseDLModel):
    def __init__(self, input_dim, hidden_dim=128):
        super().__init__(input_dim, hidden_dim)
        self.rnn = nn.RNN(input_dim, hidden_dim, batch_first=True)
        self.fc = nn.Linear(hidden_dim, 1)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        x = x.unsqueeze(1)
        x, _ = self.rnn(x)
        x = self.fc(x[:, -1, :])
        return self.sigmoid(x)

class LSTMClassifier(BaseDLModel):
    def __init__(self, input_dim, hidden_dim=128):
        super().__init__(input_dim, hidden_dim)
        self.lstm = nn.LSTM(input_dim, hidden_dim, batch_first=True, bidirectional=False)
        self.fc = nn.Linear(hidden_dim, 1)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        x = x.unsqueeze(1)
        x, _ = self.lstm(x)
        x = self.fc(x[:, -1, :])
        return self.sigmoid(x)

class BiLSTMClassifier(BaseDLModel):
    def __init__(self, input_dim, hidden_dim=128):
        super().__init__(input_dim, hidden_dim)
        self.lstm = nn.LSTM(input_dim, hidden_dim, batch_first=True, bidirectional=True)
        self.fc = nn.Linear(hidden_dim * 2, 1)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        x = x.unsqueeze(1)
        x, _ = self.lstm(x)
        x = self.fc(x[:, -1, :])
        return self.sigmoid(x)

class CNNClassifier(BaseDLModel):
    def __init__(self, input_dim, num_filters=128, kernel_size=3):
        super().__init__(input_dim)
        self.conv = nn.Conv1d(1, num_filters, kernel_size)
        self.fc = nn.Linear(num_filters, 1)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        x = x.unsqueeze(1)
        x = torch.relu(self.conv(x))
        x = self.fc(torch.max(x, dim=2)[0])
        return self.sigmoid(x)

class BiGRUClassifier(BaseDLModel):
    def __init__(self, input_dim, hidden_dim=128):
        super().__init__(input_dim, hidden_dim)
        self.gru = nn.GRU(input_dim, hidden_dim, batch_first=True, bidirectional=True)
        self.fc = nn.Linear(hidden_dim * 2, 1)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        x = x.unsqueeze(1)
        x, _ = self.gru(x)
        x = self.fc(x[:, -1, :])
        return self.sigmoid(x)

class GRUClassifier(BaseDLModel):
    def __init__(self, input_dim, hidden_dim=128):
        super().__init__(input_dim, hidden_dim)
        self.gru = nn.GRU(input_dim, hidden_dim, batch_first=True)
        self.fc = nn.Linear(hidden_dim, 1)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        x = x.unsqueeze(1)
        x, _ = self.gru(x)
        x = self.fc(x[:, -1, :])
        return self.sigmoid(x)

embeddings = {
    'TF-IDF': (X_val_tfidf, X_test_tfidf),
    'GloVe': (X_val_glove, X_test_glove),
    'SBERT': (X_val_sbert, X_test_sbert),
}

dl_models = {
    'RNN': RNNClassifier,
    'LSTM': LSTMClassifier,
    'BiLSTM': BiLSTMClassifier,
    'CNN': CNNClassifier,
    'GRU': GRUClassifier,
    'BiGRU': BiGRUClassifier,
}
def train_and_evaluate(model, X_train, X_test, y_train, y_test, p_test, epochs=10, lr=2e-4, batch_size=16):
    criterion = nn.BCELoss()
    optimizer = optim.Adam(model.parameters(), lr=lr)

    X_train_tensor = torch.tensor(X_train, dtype=torch.float32).to(device)
    y_train_tensor = torch.tensor(y_train, dtype=torch.float32).to(device)
    X_test_tensor = torch.tensor(X_test, dtype=torch.float32).to(device)
    y_test_tensor = torch.tensor(y_test, dtype=torch.float32).to(device)

    train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)

    model.to(device)

    for epoch in range(epochs):
        model.train()
        total_loss = 0
        for batch in train_loader:
            x_batch, y_batch = batch
            optimizer.zero_grad()
            y_pred = model(x_batch).squeeze()
            loss = criterion(y_pred, y_batch)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
        print(f"Epoch {epoch+1} Loss: {total_loss/len(train_loader):.4f}")

    model.eval()
    with torch.no_grad():
        y_pred_probs = model(X_test_tensor).cpu().numpy().squeeze()
        y_pred = (y_pred_probs >= 0.5).astype(int)

    accuracy = accuracy_score(y_test, y_pred)
    spearman_corr, _ = spearmanr(p_test, y_pred_probs)

    print(f"Accuracy: {accuracy:.4f}")
    print(f"Spearman Correlation: {spearman_corr:.4f}")
    print(classification_report(y_test, y_pred))

    return accuracy, spearman_corr
results = []
for emb_name, (X_train_emb, X_test_emb) in embeddings.items():
    for model_name, ModelClass in dl_models.items():
        print(f"\nTraining {model_name} with {emb_name} embeddings...")
        model = ModelClass(X_train_emb.shape[1]).to(device)
        accuracy, spearman_corr = train_and_evaluate(model, X_train_emb, X_test_emb, y_val, y_test, p_test)
        results.append({'Embedding': emb_name, 'Model': model_name, 'Accuracy': accuracy, 'Spearman Correlation': spearman_corr})

results_df = pd.DataFrame(results)
print("\nFinal Model Comparison:")
print(results_df)


Using device: cuda


<ipython-input-3-6da384173d10>:19: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  labeled_df['label'] = labeled_df['label'].apply(lambda x: 1 if x == 'Hallucination' else 0)
<ipython-input-3-6da384173d10>:26: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  labeled_df['combined_input'] = (
/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings t

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.7k [00:00<?, ?B/s]

...

Epoch 1 Loss: 0.6972
Epoch 2 Loss: 0.6872
Epoch 3 Loss: 0.6778
Epoch 4 Loss: 0.6669
Epoch 5 Loss: 0.6553
Epoch 6 Loss: 0.6407
Epoch 7 Loss: 0.6261
Epoch 8 Loss: 0.6059
Epoch 9 Loss: 0.5824
Epoch 10 Loss: 0.5548
Accuracy: 0.5953
Spearman Correlation: 0.0410
              precision    recall  f1-score   support

         0.0       0.61      0.93      0.74      1834
         1.0       0.37      0.06      0.10      1161

    accuracy                           0.60      2995
   macro avg       0.49      0.50      0.42      2995
weighted avg       0.52      0.60      0.49      2995


Training BiGRU with TF-IDF embeddings...
Epoch 1 Loss: 0.6926
Epoch 2 Loss: 0.6809
Epoch 3 Loss: 0.6686
Epoch 4 Loss: 0.6553
Epoch 5 Loss: 0.6380
Epoch 6 Loss: 0.6167
Epoch 7 Loss: 0.5899
Epoch 8 Loss: 0.5539
Epoch 9 Loss: 0.5151
Epoch 10 Loss: 0.4730
Accuracy: 0.5813
Spearman Correlation: 0.0460
              precision    recall  f1-score   support

         0.0       0.61      0.86      0.72      1834
        

/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


Epoch 1 Loss: 0.6932
Epoch 2 Loss: 0.6835
Epoch 3 Loss: 0.6799
Epoch 4 Loss: 0.6787
Epoch 5 Loss: 0.6765
Epoch 6 Loss: 0.6749
Epoch 7 Loss: 0.6750
Epoch 8 Loss: 0.6731
Epoch 9 Loss: 0.6720
Epoch 10 Loss: 0.6719
Accuracy: 0.6150
Spearman Correlation: 0.1752
              precision    recall  f1-score   support

         0.0       0.62      0.99      0.76      1834
         1.0       0.57      0.03      0.05      1161

    accuracy                           0.62      2995
   macro avg       0.59      0.51      0.40      2995
weighted avg       0.60      0.62      0.48      2995


Training BiGRU with GloVe embeddings...
Epoch 1 Loss: 0.6848
Epoch 2 Loss: 0.6795
Epoch 3 Loss: 0.6776
Epoch 4 Loss: 0.6772
Epoch 5 Loss: 0.6747
Epoch 6 Loss: 0.6736
Epoch 7 Loss: 0.6719
Epoch 8 Loss: 0.6713
Epoch 9 Loss: 0.6688
Epoch 10 Loss: 0.6682
Accuracy: 0.6154
Spearman Correlation: 0.1774
              precision    recall  f1-score   support

         0.0       0.62      0.94      0.75      1834
         

/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


Epoch 2 Loss: 0.6834
Epoch 3 Loss: 0.6771
Epoch 4 Loss: 0.6718
Epoch 5 Loss: 0.6666
Epoch 6 Loss: 0.6614
Epoch 7 Loss: 0.6550
Epoch 8 Loss: 0.6488
Epoch 9 Loss: 0.6427
Epoch 10 Loss: 0.6379
Accuracy: 0.6154
Spearman Correlation: 0.2452
              precision    recall  f1-score   support

         0.0       0.63      0.89      0.74      1834
         1.0       0.51      0.19      0.27      1161

    accuracy                           0.62      2995
   macro avg       0.57      0.54      0.51      2995
weighted avg       0.59      0.62      0.56      2995


Training BiGRU with SBERT embeddings...
Epoch 1 Loss: 0.6884
Epoch 2 Loss: 0.6791
Epoch 3 Loss: 0.6710
Epoch 4 Loss: 0.6644
Epoch 5 Loss: 0.6582
Epoch 6 Loss: 0.6496
Epoch 7 Loss: 0.6442
Epoch 8 Loss: 0.6354
Epoch 9 Loss: 0.6274
Epoch 10 Loss: 0.6216
Accuracy: 0.6197
Spearman Correlation: 0.2379
              precision    recall  f1-score   support

         0.0       0.65      0.83      0.73      1834
         1.0       0.52      0

### Transformers

#### Single Seed Transformer Models

##### Bert Large
*** bert-large-uncased
LR - 2.63e-5 Batch : 16, Epochs : 5
Accuracy: 0.6795              
Spearman Correlation: 0.3993

In [ ]:
import random
import numpy as np
import torch
import torch.nn as nn
import pandas as pd
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification, AdamW
from sklearn.metrics import accuracy_score
from scipy.stats import spearmanr
from tqdm import tqdm

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')


labeled_df = cleaned_df[cleaned_df['label'].notna()].copy()
labeled_df['label'] = labeled_df['label'].apply(lambda x: 1 if x == 'Hallucination' else 0).astype(int)

if 'p(Hallucination)' not in labeled_df.columns:
    raise ValueError("Missing `p(Hallucination)` field in dataset. Ensure probability values exist.")

labeled_df['combined_input'] = (
    labeled_df['hyp'].fillna('') + ' ' +
    labeled_df['src'].fillna('') + ' ' +
    labeled_df['tgt'].fillna('') + ' ' +
    labeled_df['ref'].fillna('') + ' ' +
    labeled_df['model'].fillna('') + ' ' +
    labeled_df['task'].fillna('')
)

val_df = labeled_df[labeled_df['Dataset'].str.contains('Val')]
test_df = labeled_df[labeled_df['Dataset'].str.contains('Test')]

X_val = val_df['combined_input'].tolist()
y_val = val_df['label'].tolist()
p_val = val_df['p(Hallucination)'].tolist()

X_test = test_df['combined_input'].tolist()
y_test = test_df['label'].tolist()
p_test = test_df['p(Hallucination)'].tolist()


class HallucinationDataset(Dataset):
    def __init__(self, texts, labels, probs, tokenizer, max_len=128):
        self.texts = texts
        self.labels = labels
        self.probs = probs
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = self.texts[idx].strip() or " "
        label = int(self.labels[idx])
        prob = float(self.probs[idx])
        encoding = self.tokenizer(
            text,
            padding='max_length',
            truncation=True,
            max_length=self.max_len,
            return_tensors='pt'
        )
        return {
            'input_ids': encoding['input_ids'].squeeze(0),
            'attention_mask': encoding['attention_mask'].squeeze(0),
            'label': torch.tensor(label, dtype=torch.long),
            'prob': torch.tensor(prob, dtype=torch.float)
        }


def train_and_evaluate_spearman(model_name, train_dataset, test_dataset, lr, batch_size, epochs, progress_bar):
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    try:
        model = AutoModelForSequenceClassification.from_pretrained(
            model_name, num_labels=2, ignore_mismatched_sizes=True
        )
    except RuntimeError as e:
        print(f"Error loading {model_name}: {e}")
        return 0, 0


    model.to(device)
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, drop_last=True)
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

    optimizer = AdamW(model.parameters(), lr=lr)

    # Training Loop
    for epoch in range(epochs):
        model.train()
        total_loss = 0
        for batch in train_loader:
            optimizer.zero_grad()
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['label'].to(device)

            # Check label range
            if labels.min() < 0 or labels.max() >= 2:
                raise ValueError(f"Label range issue detected! Labels found: {labels.unique()}")


            try:
                outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
            except Exception as e:
                print(f"Skipping training batch due to forward pass error in {model_name}: {e}")
                continue

            if outputs.logits.size(1) < 2:
                print(f" Warning: {model_name} produced logits with shape {outputs.logits.shape} in training. Skipping batch.")
                continue

            loss = outputs.loss
            loss.backward()
            optimizer.step()
            total_loss += loss.item()

        progress_bar.update(1)
        print(f"Epoch {epoch+1}/{epochs} - {model_name} Avg Loss: {total_loss/len(train_loader):.4f}")

    # Evaluation Step
    model.eval()
    all_preds = []
    all_probs = []
    true_probs = []
    with torch.no_grad():
        for batch in test_loader:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            batch_probs = batch['prob'].cpu().numpy()

            try:
                outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            except Exception as e:
                print(f" Skipping evaluation batch due to forward pass error in {model_name}: {e}")
                continue

            # Skip batch if logits shape is unexpected.
            if outputs.logits.size(1) < 2:
                print(f" Warning: {model_name} produced logits with shape {outputs.logits.shape} in evaluation. Skipping batch.")
                continue

            softmax_scores = torch.nn.functional.softmax(outputs.logits, dim=1)[:, 1]  # Score for class 1
            all_preds.extend(torch.argmax(outputs.logits, dim=1).cpu().numpy())
            all_probs.extend(softmax_scores.cpu().numpy())
            true_probs.extend(batch_probs)

    accuracy = accuracy_score(y_test, all_preds)
    print(f"Test Accuracy for {model_name}: {accuracy:.4f}")

    spearman_corr, _ = spearmanr(true_probs, all_probs)
    print(f"Spearman Correlation for {model_name}: {spearman_corr:.4f}")

    return accuracy, spearman_corr


pretrained_models = {
    'bert-large-uncased': {'lr': 2.63e-5, 'batch_size': 16, 'epochs': 5},
}


tokenizer = AutoTokenizer.from_pretrained('bert-base-uncased')
train_dataset = HallucinationDataset(X_val, y_val, p_val, tokenizer)
test_dataset = HallucinationDataset(X_test, y_test, p_test, tokenizer)

total_epochs = sum(params['epochs'] for params in pretrained_models.values())
results = []

with tqdm(total=total_epochs, desc="Overall Progress") as progress_bar:
    for model_name, params in pretrained_models.items():
        print(f"\nEvaluating model: {model_name}")
        accuracy, spearman_corr = train_and_evaluate_spearman(
            model_name, train_dataset, test_dataset,
            lr=params['lr'], batch_size=params['batch_size'], epochs=params['epochs'],
            progress_bar=progress_bar
        )
        results.append({'Model': model_name, 'Accuracy': accuracy, 'Spearman Correlation': spearman_corr})

results_df = pd.DataFrame(results)
print("\nFinal Optimized Model Results:")
print(results_df)

Using device: cuda


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

...

Epoch 4/5 - bert-large-uncased Avg Loss: 0.2977


Overall Progress: 100%|██████████| 5/5 [06:14<00:00, 73.58s/it]

Epoch 5/5 - bert-large-uncased Avg Loss: 0.1571


Overall Progress: 100%|██████████| 5/5 [07:30<00:00, 90.12s/it]

Test Accuracy for bert-large-uncased: 0.6795
Spearman Correlation for bert-large-uncased: 0.3993

Final Optimized Model Results:
                Model  Accuracy  Spearman Correlation
0  bert-large-uncased  0.679466              0.399253


##### Google Electra Large
*** google/electra-large-discriminator: {'lr': 2e-5, 'batch_size': 16, 'epochs': 5}
Accuracy: 0.692487  Spearman Correlation: 0.481698

In [ ]:
import random
import numpy as np
import torch
import torch.nn as nn
import pandas as pd
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification, AdamW
from sklearn.metrics import accuracy_score
from scipy.stats import spearmanr
from tqdm import tqdm

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

labeled_df = cleaned_df[cleaned_df['label'].notna()].copy()
labeled_df['label'] = labeled_df['label'].apply(lambda x: 1 if x == 'Hallucination' else 0).astype(int)

if 'p(Hallucination)' not in labeled_df.columns:
    raise ValueError("Missing `p(Hallucination)` field in dataset. Ensure probability values exist.")

labeled_df['combined_input'] = (
    labeled_df['hyp'].fillna('') + ' ' +
    labeled_df['src'].fillna('') + ' ' +
    labeled_df['tgt'].fillna('') + ' ' +
    labeled_df['ref'].fillna('') + ' ' +
    labeled_df['model'].fillna('') + ' ' +
    labeled_df['task'].fillna('')
)

val_df = labeled_df[labeled_df['Dataset'].str.contains('Val')]
test_df = labeled_df[labeled_df['Dataset'].str.contains('Test')]

X_val = val_df['combined_input'].tolist()
y_val = val_df['label'].tolist()
p_val = val_df['p(Hallucination)'].tolist()

X_test = test_df['combined_input'].tolist()
y_test = test_df['label'].tolist()
p_test = test_df['p(Hallucination)'].tolist()


class HallucinationDataset(Dataset):
    def __init__(self, texts, labels, probs, tokenizer, max_len=128):
        self.texts = texts
        self.labels = labels
        self.probs = probs
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = self.texts[idx].strip() or " "
        label = int(self.labels[idx])
        prob = float(self.probs[idx])
        encoding = self.tokenizer(
            text,
            padding='max_length',
            truncation=True,
            max_length=self.max_len,
            return_tensors='pt'
        )
        return {
            'input_ids': encoding['input_ids'].squeeze(0),
            'attention_mask': encoding['attention_mask'].squeeze(0),
            'label': torch.tensor(label, dtype=torch.long),
            'prob': torch.tensor(prob, dtype=torch.float)
        }

def train_and_evaluate_spearman(model_name, train_dataset, test_dataset, lr, batch_size, epochs, progress_bar):
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    try:
        model = AutoModelForSequenceClassification.from_pretrained(
            model_name, num_labels=2, ignore_mismatched_sizes=True
        )
    except RuntimeError as e:
        print(f"⚠️ Error loading {model_name}: {e}")
        return 0, 0


    model.to(device)

    # Use drop_last for training to ensure full batches.
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, drop_last=True)
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

    optimizer = AdamW(model.parameters(), lr=lr)

    # Training Loop
    for epoch in range(epochs):
        model.train()
        total_loss = 0
        for batch in train_loader:
            optimizer.zero_grad()
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['label'].to(device)

            # Check label range
            if labels.min() < 0 or labels.max() >= 2:
                raise ValueError(f"Label range issue detected! Labels found: {labels.unique()}")

            # Wrap the forward pass in try/except to catch problematic batches.
            try:
                outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
            except Exception as e:
                print(f" Skipping training batch due to forward pass error in {model_name}: {e}")
                continue

            # Skip batch if logits shape is not as expected.
            if outputs.logits.size(1) < 2:
                print(f" Warning: {model_name} produced logits with shape {outputs.logits.shape} in training. Skipping batch.")
                continue

            loss = outputs.loss
            loss.backward()
            optimizer.step()
            total_loss += loss.item()

        progress_bar.update(1)
        print(f"Epoch {epoch+1}/{epochs} - {model_name} Avg Loss: {total_loss/len(train_loader):.4f}")

    # Evaluation Step
    model.eval()
    all_preds = []
    all_probs = []
    true_probs = []
    with torch.no_grad():
        for batch in test_loader:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            batch_probs = batch['prob'].cpu().numpy()

            try:
                outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            except Exception as e:
                print(f"Skipping evaluation batch due to forward pass error in {model_name}: {e}")
                continue

            # Skip batch if logits shape is unexpected.
            if outputs.logits.size(1) < 2:
                print(f"Warning: {model_name} produced logits with shape {outputs.logits.shape} in evaluation. Skipping batch.")
                continue

            softmax_scores = torch.nn.functional.softmax(outputs.logits, dim=1)[:, 1]  # Score for class 1
            all_preds.extend(torch.argmax(outputs.logits, dim=1).cpu().numpy())
            all_probs.extend(softmax_scores.cpu().numpy())
            true_probs.extend(batch_probs)

    accuracy = accuracy_score(y_test, all_preds)
    print(f"Test Accuracy for {model_name}: {accuracy:.4f}")

    spearman_corr, _ = spearmanr(true_probs, all_probs)
    print(f"Spearman Correlation for {model_name}: {spearman_corr:.4f}")

    return accuracy, spearman_corr

pretrained_models = {
    'google/electra-large-discriminator': {'lr': 2e-5, 'batch_size': 16, 'epochs': 5},
}

tokenizer = AutoTokenizer.from_pretrained('google/electra-large-discriminator')
train_dataset = HallucinationDataset(X_val, y_val, p_val, tokenizer)
test_dataset = HallucinationDataset(X_test, y_test, p_test, tokenizer)

total_epochs = sum(params['epochs'] for params in pretrained_models.values())
results = []

with tqdm(total=total_epochs, desc="Overall Progress") as progress_bar:
    for model_name, params in pretrained_models.items():
        print(f"\nEvaluating model: {model_name}")
        accuracy, spearman_corr = train_and_evaluate_spearman(
            model_name, train_dataset, test_dataset,
            lr=params['lr'], batch_size=params['batch_size'], epochs=params['epochs'],
            progress_bar=progress_bar
        )
        results.append({'Model': model_name, 'Accuracy': accuracy, 'Spearman Correlation': spearman_corr})

results_df = pd.DataFrame(results)
print("\nFinal Optimized Model Results:")
print(results_df)


Using device: cuda


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/668 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

...

Epoch 4/5 - google/electra-large-discriminator Avg Loss: 0.4136


Overall Progress: 100%|██████████| 5/5 [06:24<00:00, 75.56s/it]

Epoch 5/5 - google/electra-large-discriminator Avg Loss: 0.3710


Overall Progress: 100%|██████████| 5/5 [07:39<00:00, 91.96s/it]

Test Accuracy for google/electra-large-discriminator: 0.6925
Spearman Correlation for google/electra-large-discriminator: 0.4817

Final Optimized Model Results:
                                Model  Accuracy  Spearman Correlation
0  google/electra-large-discriminator  0.692487              0.481698


##### DistilBERT Base
*** distilbert-base-uncased: {'lr': 2e-5, 'batch_size': 16, 'epochs': 5}
Accuracy: 0.656093  Spearman Correlation: 0.343

In [ ]:
import random
import numpy as np
import torch
import torch.nn as nn
import pandas as pd
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification, AdamW
from sklearn.metrics import accuracy_score
from scipy.stats import spearmanr
from tqdm import tqdm


SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False


device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

labeled_df = cleaned_df[cleaned_df['label'].notna()].copy()
labeled_df['label'] = labeled_df['label'].apply(lambda x: 1 if x == 'Hallucination' else 0).astype(int)

if 'p(Hallucination)' not in labeled_df.columns:
    raise ValueError("Missing `p(Hallucination)` field in dataset. Ensure probability values exist.")

labeled_df['combined_input'] = (
    labeled_df['hyp'].fillna('') + ' ' +
    labeled_df['src'].fillna('') + ' ' +
    labeled_df['tgt'].fillna('') + ' ' +
    labeled_df['ref'].fillna('') + ' ' +
    labeled_df['model'].fillna('') + ' ' +
    labeled_df['task'].fillna('')
)

val_df = labeled_df[labeled_df['Dataset'].str.contains('Val')]
test_df = labeled_df[labeled_df['Dataset'].str.contains('Test')]

X_val = val_df['combined_input'].tolist()
y_val = val_df['label'].tolist()
p_val = val_df['p(Hallucination)'].tolist()

X_test = test_df['combined_input'].tolist()
y_test = test_df['label'].tolist()
p_test = test_df['p(Hallucination)'].tolist()

class HallucinationDataset(Dataset):
    def __init__(self, texts, labels, probs, tokenizer, max_len=128):
        self.texts = texts
        self.labels = labels
        self.probs = probs
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = self.texts[idx].strip() or " "
        label = int(self.labels[idx])
        prob = float(self.probs[idx])
        encoding = self.tokenizer(
            text,
            padding='max_length',
            truncation=True,
            max_length=self.max_len,
            return_tensors='pt'
        )
        return {
            'input_ids': encoding['input_ids'].squeeze(0),
            'attention_mask': encoding['attention_mask'].squeeze(0),
            'label': torch.tensor(label, dtype=torch.long),
            'prob': torch.tensor(prob, dtype=torch.float)
        }

def train_and_evaluate_spearman(model_name, train_dataset, test_dataset, lr, batch_size, epochs, progress_bar):
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    try:
        model = AutoModelForSequenceClassification.from_pretrained(
            model_name, num_labels=2, ignore_mismatched_sizes=True
        )
    except RuntimeError as e:
        print(f"⚠️ Error loading {model_name}: {e}")
        return 0, 0


    model.to(device)

    # Use drop_last for training to ensure full batches.
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, drop_last=True)
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

    optimizer = AdamW(model.parameters(), lr=lr)

    # Training Loop
    for epoch in range(epochs):
        model.train()
        total_loss = 0
        for batch in train_loader:
            optimizer.zero_grad()
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['label'].to(device)

            # Check label range
            if labels.min() < 0 or labels.max() >= 2:
                raise ValueError(f"Label range issue detected! Labels found: {labels.unique()}")

            # Wrap the forward pass in try/except to catch problematic batches.
            try:
                outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
            except Exception as e:
                print(f" Skipping training batch due to forward pass error in {model_name}: {e}")
                continue

            # Skip batch if logits shape is not as expected.
            if outputs.logits.size(1) < 2:
                print(f" Warning: {model_name} produced logits with shape {outputs.logits.shape} in training. Skipping batch.")
                continue

            loss = outputs.loss
            loss.backward()
            optimizer.step()
            total_loss += loss.item()

        progress_bar.update(1)
        print(f"Epoch {epoch+1}/{epochs} - {model_name} Avg Loss: {total_loss/len(train_loader):.4f}")

    # Evaluation Step
    model.eval()
    all_preds = []
    all_probs = []
    true_probs = []
    with torch.no_grad():
        for batch in test_loader:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            batch_probs = batch['prob'].cpu().numpy()

            try:
                outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            except Exception as e:
                print(f"Skipping evaluation batch due to forward pass error in {model_name}: {e}")
                continue

            # Skip batch if logits shape is unexpected.
            if outputs.logits.size(1) < 2:
                print(f"Warning: {model_name} produced logits with shape {outputs.logits.shape} in evaluation. Skipping batch.")
                continue

            softmax_scores = torch.nn.functional.softmax(outputs.logits, dim=1)[:, 1]  # Score for class 1
            all_preds.extend(torch.argmax(outputs.logits, dim=1).cpu().numpy())
            all_probs.extend(softmax_scores.cpu().numpy())
            true_probs.extend(batch_probs)

    accuracy = accuracy_score(y_test, all_preds)
    print(f"Test Accuracy for {model_name}: {accuracy:.4f}")

    spearman_corr, _ = spearmanr(true_probs, all_probs)
    print(f"Spearman Correlation for {model_name}: {spearman_corr:.4f}")

    return accuracy, spearman_corr

pretrained_models = {
    'distilbert-base-uncased': {'lr': 2e-5, 'batch_size': 16, 'epochs': 5},
}

tokenizer = AutoTokenizer.from_pretrained('distilbert-base-uncased')
train_dataset = HallucinationDataset(X_val, y_val, p_val, tokenizer)
test_dataset = HallucinationDataset(X_test, y_test, p_test, tokenizer)

total_epochs = sum(params['epochs'] for params in pretrained_models.values())
results = []

with tqdm(total=total_epochs, desc="Overall Progress") as progress_bar:
    for model_name, params in pretrained_models.items():
        print(f"\nEvaluating model: {model_name}")
        accuracy, spearman_corr = train_and_evaluate_spearman(
            model_name, train_dataset, test_dataset,
            lr=params['lr'], batch_size=params['batch_size'], epochs=params['epochs'],
            progress_bar=progress_bar
        )
        results.append({'Model': model_name, 'Accuracy': accuracy, 'Spearman Correlation': spearman_corr})

results_df = pd.DataFrame(results)
print("\nFinal Optimized Model Results:")
print(results_df)

Using device: cuda


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

...

Epoch 4/5 - distilbert-base-uncased Avg Loss: 0.2951


Overall Progress: 100%|██████████| 5/5 [00:27<00:00,  5.24s/it]

Epoch 5/5 - distilbert-base-uncased Avg Loss: 0.1107


Overall Progress: 100%|██████████| 5/5 [00:33<00:00,  6.64s/it]

Test Accuracy for distilbert-base-uncased: 0.6561
Spearman Correlation for distilbert-base-uncased: 0.3430

Final Optimized Model Results:
                     Model  Accuracy  Spearman Correlation
0  distilbert-base-uncased  0.656093                 0.343


##### RoBERTA Large MNLI
roberta-large-mnli: {'lr': 2e-05, 'batch_size': 8, 'epochs': 5}
Accuracy : 0.753589           Spearman Correlation:   0.550234

In [ ]:
import random
import numpy as np
import torch
import torch.nn as nn
import pandas as pd
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification, AdamW
from sklearn.metrics import accuracy_score
from scipy.stats import spearmanr
from tqdm import tqdm

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False


device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')


labeled_df = cleaned_df[cleaned_df['label'].notna()].copy()
labeled_df['label'] = labeled_df['label'].apply(lambda x: 1 if x == 'Hallucination' else 0).astype(int)

if 'p(Hallucination)' not in labeled_df.columns:
    raise ValueError("Missing `p(Hallucination)` field in dataset. Ensure probability values exist.")

labeled_df['combined_input'] = (
    labeled_df['hyp'].fillna('') + ' ' +
    labeled_df['src'].fillna('') + ' ' +
    labeled_df['tgt'].fillna('') + ' ' +
    labeled_df['ref'].fillna('') + ' ' +
    labeled_df['model'].fillna('') + ' ' +
    labeled_df['task'].fillna('')
)

val_df = labeled_df[labeled_df['Dataset'].str.contains('Val')]
test_df = labeled_df[labeled_df['Dataset'].str.contains('Test')]

X_val = val_df['combined_input'].tolist()
y_val = val_df['label'].tolist()
p_val = val_df['p(Hallucination)'].tolist()

X_test = test_df['combined_input'].tolist()
y_test = test_df['label'].tolist()
p_test = test_df['p(Hallucination)'].tolist()


class HallucinationDataset(Dataset):
    def __init__(self, texts, labels, probs, tokenizer, max_len=128):
        self.texts = texts
        self.labels = labels
        self.probs = probs
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = self.texts[idx].strip() or " "
        label = int(self.labels[idx])
        prob = float(self.probs[idx])
        encoding = self.tokenizer(
            text,
            padding='max_length',
            truncation=True,
            max_length=self.max_len,
            return_tensors='pt'
        )
        return {
            'input_ids': encoding['input_ids'].squeeze(0),
            'attention_mask': encoding['attention_mask'].squeeze(0),
            'label': torch.tensor(label, dtype=torch.long),
            'prob': torch.tensor(prob, dtype=torch.float)
        }


def train_and_evaluate_spearman(model_name, train_dataset, test_dataset, lr, batch_size, epochs, progress_bar):
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    try:
        model = AutoModelForSequenceClassification.from_pretrained(
            model_name, num_labels=2, ignore_mismatched_sizes=True
        )
    except RuntimeError as e:
        print(f"Error loading {model_name}: {e}")
        return 0, 0


    model.to(device)


    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, drop_last=True)
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

    optimizer = AdamW(model.parameters(), lr=lr)

    # Training Loop
    for epoch in range(epochs):
        model.train()
        total_loss = 0
        for batch in train_loader:
            optimizer.zero_grad()
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['label'].to(device)

            # Check label range
            if labels.min() < 0 or labels.max() >= 2:
                raise ValueError(f"Label range issue detected! Labels found: {labels.unique()}")

            # Wrap the forward pass in try/except to catch problematic batches.
            try:
                outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
            except Exception as e:
                print(f"Skipping training batch due to forward pass error in {model_name}: {e}")
                continue

            # Skip batch if logits shape is not as expected.
            if outputs.logits.size(1) < 2:
                print(f"Warning: {model_name} produced logits with shape {outputs.logits.shape} in training. Skipping batch.")
                continue

            loss = outputs.loss
            loss.backward()
            optimizer.step()
            total_loss += loss.item()

        progress_bar.update(1)
        print(f"Epoch {epoch+1}/{epochs} - {model_name} Avg Loss: {total_loss/len(train_loader):.4f}")

    # Evaluation Step
    model.eval()
    all_preds = []
    all_probs = []
    true_probs = []
    with torch.no_grad():
        for batch in test_loader:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            batch_probs = batch['prob'].cpu().numpy()

            try:
                outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            except Exception as e:
                print(f"⚠️ Skipping evaluation batch due to forward pass error in {model_name}: {e}")
                continue

            # Skip batch if logits shape is unexpected.
            if outputs.logits.size(1) < 2:
                print(f"⚠️ Warning: {model_name} produced logits with shape {outputs.logits.shape} in evaluation. Skipping batch.")
                continue

            softmax_scores = torch.nn.functional.softmax(outputs.logits, dim=1)[:, 1]  # Score for class 1
            all_preds.extend(torch.argmax(outputs.logits, dim=1).cpu().numpy())
            all_probs.extend(softmax_scores.cpu().numpy())
            true_probs.extend(batch_probs)

    accuracy = accuracy_score(y_test, all_preds)
    print(f"Test Accuracy for {model_name}: {accuracy:.4f}")

    spearman_corr, _ = spearmanr(true_probs, all_probs)
    print(f"Spearman Correlation for {model_name}: {spearman_corr:.4f}")

    return accuracy, spearman_corr


pretrained_models = {
    'roberta-large-mnli': {'lr': 2e-05, 'batch_size': 8, 'epochs': 5},
}

tokenizer = AutoTokenizer.from_pretrained('roberta-large-mnli')
train_dataset = HallucinationDataset(X_val, y_val, p_val, tokenizer)
test_dataset = HallucinationDataset(X_test, y_test, p_test, tokenizer)

total_epochs = sum(params['epochs'] for params in pretrained_models.values())
results = []

with tqdm(total=total_epochs, desc="Overall Progress") as progress_bar:
    for model_name, params in pretrained_models.items():
        print(f"\nEvaluating model: {model_name}")
        accuracy, spearman_corr = train_and_evaluate_spearman(
            model_name, train_dataset, test_dataset,
            lr=params['lr'], batch_size=params['batch_size'], epochs=params['epochs'],
            progress_bar=progress_bar
        )
        results.append({'Model': model_name, 'Accuracy': accuracy, 'Spearman Correlation': spearman_corr})

results_df = pd.DataFrame(results)
print("\nFinal Optimized Model Results:")
print(results_df)

Using device: cuda


tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/688 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

...

Epoch 4/5 - roberta-large-mnli Avg Loss: 0.1209


Overall Progress: 100%|██████████| 5/5 [03:27<00:00, 39.26s/it]

Epoch 5/5 - roberta-large-mnli Avg Loss: 0.0925


Overall Progress: 100%|██████████| 5/5 [04:00<00:00, 48.16s/it]

Test Accuracy for roberta-large-mnli: 0.7536
Spearman Correlation for roberta-large-mnli: 0.5502

Final Optimized Model Results:
                Model  Accuracy  Spearman Correlation
0  roberta-large-mnli  0.753589              0.550234


##### Microsoft DeBERTA v3 Large (BEST MODEL)
*** Model: microsoft/deberta-v3-large: {'lr': 2e-05, 'batch_size': 16, 'epochs': 5}
Accuracy: 0.765609  Spearman Correlation: 0.627719

In [ ]:
import random
import numpy as np
import torch
import torch.nn as nn
import pandas as pd
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification, AdamW
from sklearn.metrics import accuracy_score
from scipy.stats import spearmanr
from tqdm import tqdm

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')


labeled_df = cleaned_df[cleaned_df['label'].notna()].copy()
labeled_df['label'] = labeled_df['label'].apply(lambda x: 1 if x == 'Hallucination' else 0).astype(int)

if 'p(Hallucination)' not in labeled_df.columns:
    raise ValueError("Missing `p(Hallucination)` field in dataset. Ensure probability values exist.")

labeled_df['combined_input'] = (
    labeled_df['hyp'].fillna('') + ' ' +
    labeled_df['src'].fillna('') + ' ' +
    labeled_df['tgt'].fillna('') + ' ' +
    labeled_df['ref'].fillna('') + ' ' +
    labeled_df['model'].fillna('') + ' ' +
    labeled_df['task'].fillna('')
)

val_df = labeled_df[labeled_df['Dataset'].str.contains('Val')]
test_df = labeled_df[labeled_df['Dataset'].str.contains('Test')]

X_val = val_df['combined_input'].tolist()
y_val = val_df['label'].tolist()
p_val = val_df['p(Hallucination)'].tolist()

X_test = test_df['combined_input'].tolist()
y_test = test_df['label'].tolist()
p_test = test_df['p(Hallucination)'].tolist()

class HallucinationDataset(Dataset):
    def __init__(self, texts, labels, probs, tokenizer, max_len=128):
        self.texts = texts
        self.labels = labels
        self.probs = probs
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = self.texts[idx].strip() or " "
        label = int(self.labels[idx])
        prob = float(self.probs[idx])
        encoding = self.tokenizer(
            text,
            padding='max_length',
            truncation=True,
            max_length=self.max_len,
            return_tensors='pt'
        )
        return {
            'input_ids': encoding['input_ids'].squeeze(0),
            'attention_mask': encoding['attention_mask'].squeeze(0),
            'label': torch.tensor(label, dtype=torch.long),
            'prob': torch.tensor(prob, dtype=torch.float)
        }

def train_and_evaluate_spearman(model_name, train_dataset, test_dataset, lr, batch_size, epochs, progress_bar):
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    try:
        model = AutoModelForSequenceClassification.from_pretrained(
            model_name, num_labels=2, ignore_mismatched_sizes=True
        )
    except RuntimeError as e:
        print(f"Error loading {model_name}: {e}")
        return 0, 0


    model.to(device)

    # Use drop_last for training to ensure full batches.
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, drop_last=True)
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

    optimizer = AdamW(model.parameters(), lr=lr)

    # Training Loop
    for epoch in range(epochs):
        model.train()
        total_loss = 0
        for batch in train_loader:
            optimizer.zero_grad()
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['label'].to(device)

            # Check label range
            if labels.min() < 0 or labels.max() >= 2:
                raise ValueError(f"Label range issue detected! Labels found: {labels.unique()}")

            # Wrap the forward pass in try/except to catch problematic batches.
            try:
                outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
            except Exception as e:
                print(f"Skipping training batch due to forward pass error in {model_name}: {e}")
                continue

            # Skip batch if logits shape is not as expected.
            if outputs.logits.size(1) < 2:
                print(f"Warning: {model_name} produced logits with shape {outputs.logits.shape} in training. Skipping batch.")
                continue

            loss = outputs.loss
            loss.backward()
            optimizer.step()
            total_loss += loss.item()

        progress_bar.update(1)
        print(f"Epoch {epoch+1}/{epochs} - {model_name} Avg Loss: {total_loss/len(train_loader):.4f}")

    # Evaluation Step
    model.eval()
    all_preds = []
    all_probs = []
    true_probs = []
    with torch.no_grad():
        for batch in test_loader:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            batch_probs = batch['prob'].cpu().numpy()

            try:
                outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            except Exception as e:
                print(f"⚠️ Skipping evaluation batch due to forward pass error in {model_name}: {e}")
                continue

            # Skip batch if logits shape is unexpected.
            if outputs.logits.size(1) < 2:
                print(f"⚠️ Warning: {model_name} produced logits with shape {outputs.logits.shape} in evaluation. Skipping batch.")
                continue

            softmax_scores = torch.nn.functional.softmax(outputs.logits, dim=1)[:, 1]  # Score for class 1
            all_preds.extend(torch.argmax(outputs.logits, dim=1).cpu().numpy())
            all_probs.extend(softmax_scores.cpu().numpy())
            true_probs.extend(batch_probs)

    accuracy = accuracy_score(y_test, all_preds)
    print(f"Test Accuracy for {model_name}: {accuracy:.4f}")

    spearman_corr, _ = spearmanr(true_probs, all_probs)
    print(f"Spearman Correlation for {model_name}: {spearman_corr:.4f}")

    return accuracy, spearman_corr


pretrained_models = {
     'microsoft/deberta-v3-large': {'lr': 2e-05, 'batch_size': 16, 'epochs': 5},
}

tokenizer = AutoTokenizer.from_pretrained('microsoft/deberta-v3-large')
train_dataset = HallucinationDataset(X_val, y_val, p_val, tokenizer)
test_dataset = HallucinationDataset(X_test, y_test, p_test, tokenizer)

total_epochs = sum(params['epochs'] for params in pretrained_models.values())
results = []

with tqdm(total=total_epochs, desc="Overall Progress") as progress_bar:
    for model_name, params in pretrained_models.items():
        print(f"\nEvaluating model: {model_name}")
        accuracy, spearman_corr = train_and_evaluate_spearman(
            model_name, train_dataset, test_dataset,
            lr=params['lr'], batch_size=params['batch_size'], epochs=params['epochs'],
            progress_bar=progress_bar
        )
        results.append({'Model': model_name, 'Accuracy': accuracy, 'Spearman Correlation': spearman_corr})

results_df = pd.DataFrame(results)
print("\nFinal Optimized Model Results:")
print(results_df)

Using device: cuda


tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/580 [00:00<?, ?B/s]

spm.model:   0%|          | 0.00/2.46M [00:00<?, ?B/s]

/usr/local/lib/python3.11/dist-packages/transformers/convert_slow_tokenizer.py:561: UserWarning: The sentencepiece tokenizer that you are converting to a fast tokenizer uses the byte fallback option which is not implemented in the fast tokenizers. In practice this means that the fast version of the tokenizer can produce unknown tokens whereas the sentencepiece version would have converted these unknown tokens into a sequence of byte tokens matching the original piece of text.
  warnings.warn(
Overall Progress:   0%|          | 0/5 [00:00<?, ?it/s]

...

Epoch 4/5 - microsoft/deberta-v3-large Avg Loss: 0.2039


Overall Progress: 100%|██████████| 5/5 [07:25<00:00, 88.01s/it]

Epoch 5/5 - microsoft/deberta-v3-large Avg Loss: 0.0751


Overall Progress: 100%|██████████| 5/5 [08:51<00:00, 106.27s/it]

Test Accuracy for microsoft/deberta-v3-large: 0.7656
Spearman Correlation for microsoft/deberta-v3-large: 0.6277

Final Optimized Model Results:
                        Model  Accuracy  Spearman Correlation
0  microsoft/deberta-v3-large  0.765609              0.627719


#### Multi Seed Averaging For Stability

##### DistilBERT Base

In [ ]:
import random
import numpy as np
import torch
import gc
from torch.utils.data import DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification, AdamW
from sklearn.metrics import accuracy_score
from scipy.stats import spearmanr
from tqdm import tqdm
import pandas as pd

# Check for GPU availability
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

# Ensure Labels are Integers and Within Range
labeled_df = cleaned_df[cleaned_df['label'].notna()].copy()
labeled_df['label'] = labeled_df['label'].apply(lambda x: 1 if x == 'Hallucination' else 0).astype(int)

if 'p(Hallucination)' not in labeled_df.columns:
    raise ValueError("Missing `p(Hallucination)` field.")

labeled_df['combined_input'] = (
    labeled_df['hyp'].fillna('') + ' ' +
    labeled_df['src'].fillna('') + ' ' +
    labeled_df['tgt'].fillna('') + ' ' +
    labeled_df['ref'].fillna('') + ' ' +
    labeled_df['model'].fillna('') + ' ' +
    labeled_df['task'].fillna('')
)

train_df = labeled_df[labeled_df['Dataset'].str.contains('Val')]
test_df = labeled_df[labeled_df['Dataset'].str.contains('Test')]

X_train = train_df['combined_input'].tolist()
y_train = train_df['label'].tolist()
p_train = train_df['p(Hallucination)'].tolist()

X_test = test_df['combined_input'].tolist()
y_test = test_df['label'].tolist()
p_test = test_df['p(Hallucination)'].tolist()

class HallucinationDataset(torch.utils.data.Dataset):
    def __init__(self, texts, labels, probs, tokenizer, max_len=128):
        self.texts = texts
        self.labels = labels
        self.probs = probs
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        encoding = self.tokenizer(
            self.texts[idx], padding='max_length', truncation=True,
            max_length=self.max_len, return_tensors='pt'
        )
        return {
            'input_ids': encoding['input_ids'].squeeze(0),
            'attention_mask': encoding['attention_mask'].squeeze(0),
            'label': torch.tensor(self.labels[idx], dtype=torch.long),
            'prob': torch.tensor(self.probs[idx], dtype=torch.float)
        }

def create_datasets(model_name):
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    train_dataset = HallucinationDataset(X_train, y_train, p_train, tokenizer)
    test_dataset = HallucinationDataset(X_test, y_test, p_test, tokenizer)
    return train_dataset, test_dataset

def train_and_evaluate(model_name, params, seeds=[42, 52, 62]):
    acc_scores, spearman_scores = [], []
    for seed in seeds:
        random.seed(seed)
        np.random.seed(seed)
        torch.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

        train_dataset, test_dataset = create_datasets(model_name)

        model = AutoModelForSequenceClassification.from_pretrained(
            model_name, num_labels=2, ignore_mismatched_sizes=True
        ).to(device)

        loader = DataLoader(train_dataset, batch_size=params['batch_size'], shuffle=True, drop_last=True)
        optimizer = AdamW(model.parameters(), lr=params['lr'])

        model.train()
        for epoch in range(params['epochs']):
            total_loss = 0
            for batch in tqdm(loader, desc=f"{model_name} Epoch {epoch+1} Seed {seed}"):
                optimizer.zero_grad()
                outputs = model(
                    input_ids=batch['input_ids'].to(device),
                    attention_mask=batch['attention_mask'].to(device),
                    labels=batch['label'].to(device)
                )
                loss = outputs.loss
                loss.backward()
                optimizer.step()
                total_loss += loss.item()

            gc.collect(); torch.cuda.empty_cache()

        model.eval()
        test_loader = DataLoader(test_dataset, batch_size=params['batch_size'], shuffle=False)
        probs, preds, true_labels, true_probs = [], [], [], []

        with torch.no_grad():
            for batch in test_loader:
                outputs = model(
                    input_ids=batch['input_ids'].to(device),
                    attention_mask=batch['attention_mask'].to(device)
                )
                batch_probs = torch.softmax(outputs.logits, dim=1)[:, 1].detach().cpu().numpy()
                batch_preds = (batch_probs > 0.5).astype(int)

                probs.extend(batch_probs)
                preds.extend(batch_preds)
                true_labels.extend(batch['label'].cpu().numpy())
                true_probs.extend(batch['prob'].cpu().numpy())

        acc = accuracy_score(true_labels, preds)
        spearman_corr, _ = spearmanr(true_probs, probs)
        acc_scores.append(acc)
        spearman_scores.append(spearman_corr)

    print(f"\n{model_name} Results:")
    print(f"Average Accuracy: {np.mean(acc_scores):.4f} ± {np.std(acc_scores):.4f}")
    print(f"Average Spearman Correlation: {np.mean(spearman_scores):.4f} ± {np.std(spearman_scores):.4f}")


models_params = {
     'distilbert-base-uncased': {'lr': 2e-5, 'batch_size': 16, 'epochs': 5}
}

for model_name, params in models_params.items():
    train_and_evaluate(model_name, params)

Using device: cuda


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/usr/local/lib/python3.11/dist-packages/transformers/optimization.py:591: FutureWarning: This implementation of AdamW is deprecated and will be removed in a future version. Use the PyTorch implementation torch.optim.AdamW instead, or set `no_deprecation_warning=True` to disable this warning
  warnings.warn(
distilbert-base-uncased Epoch 5 Seed 42: 100%|██████████| 62/62 [00:11<00:00,  5.54it/s]
Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You sho


distilbert-base-uncased Results:
Average Accuracy: 0.6495 ± 0.0036
Average Spearman Correlation: 0.3468 ± 0.0291


##### Bert Large

In [ ]:
import random
import numpy as np
import torch
import gc
from torch.utils.data import DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification, AdamW
from sklearn.metrics import accuracy_score
from scipy.stats import spearmanr
from tqdm import tqdm
import pandas as pd

# Check for GPU availability
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

# Ensure Labels are Integers and Within Range
labeled_df = cleaned_df[cleaned_df['label'].notna()].copy()
labeled_df['label'] = labeled_df['label'].apply(lambda x: 1 if x == 'Hallucination' else 0).astype(int)

if 'p(Hallucination)' not in labeled_df.columns:
    raise ValueError("Missing `p(Hallucination)` field.")

labeled_df['combined_input'] = (
    labeled_df['hyp'].fillna('') + ' ' +
    labeled_df['src'].fillna('') + ' ' +
    labeled_df['tgt'].fillna('') + ' ' +
    labeled_df['ref'].fillna('') + ' ' +
    labeled_df['model'].fillna('') + ' ' +
    labeled_df['task'].fillna('')
)

train_df = labeled_df[labeled_df['Dataset'].str.contains('Val')]
test_df = labeled_df[labeled_df['Dataset'].str.contains('Test')]

X_train = train_df['combined_input'].tolist()
y_train = train_df['label'].tolist()
p_train = train_df['p(Hallucination)'].tolist()

X_test = test_df['combined_input'].tolist()
y_test = test_df['label'].tolist()
p_test = test_df['p(Hallucination)'].tolist()

class HallucinationDataset(torch.utils.data.Dataset):
    def __init__(self, texts, labels, probs, tokenizer, max_len=128):
        self.texts = texts
        self.labels = labels
        self.probs = probs
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        encoding = self.tokenizer(
            self.texts[idx], padding='max_length', truncation=True,
            max_length=self.max_len, return_tensors='pt'
        )
        return {
            'input_ids': encoding['input_ids'].squeeze(0),
            'attention_mask': encoding['attention_mask'].squeeze(0),
            'label': torch.tensor(self.labels[idx], dtype=torch.long),
            'prob': torch.tensor(self.probs[idx], dtype=torch.float)
        }

def create_datasets(model_name):
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    train_dataset = HallucinationDataset(X_train, y_train, p_train, tokenizer)
    test_dataset = HallucinationDataset(X_test, y_test, p_test, tokenizer)
    return train_dataset, test_dataset

def train_and_evaluate(model_name, params, seeds=[42, 52, 62]):
    acc_scores, spearman_scores = [], []
    for seed in seeds:
        random.seed(seed)
        np.random.seed(seed)
        torch.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

        train_dataset, test_dataset = create_datasets(model_name)

        model = AutoModelForSequenceClassification.from_pretrained(
            model_name, num_labels=2, ignore_mismatched_sizes=True
        ).to(device)

        loader = DataLoader(train_dataset, batch_size=params['batch_size'], shuffle=True, drop_last=True)
        optimizer = AdamW(model.parameters(), lr=params['lr'])

        model.train()
        for epoch in range(params['epochs']):
            total_loss = 0
            for batch in tqdm(loader, desc=f"{model_name} Epoch {epoch+1} Seed {seed}"):
                optimizer.zero_grad()
                outputs = model(
                    input_ids=batch['input_ids'].to(device),
                    attention_mask=batch['attention_mask'].to(device),
                    labels=batch['label'].to(device)
                )
                loss = outputs.loss
                loss.backward()
                optimizer.step()
                total_loss += loss.item()

            gc.collect(); torch.cuda.empty_cache()

        model.eval()
        test_loader = DataLoader(test_dataset, batch_size=params['batch_size'], shuffle=False)
        probs, preds, true_labels, true_probs = [], [], [], []

        with torch.no_grad():
            for batch in test_loader:
                outputs = model(
                    input_ids=batch['input_ids'].to(device),
                    attention_mask=batch['attention_mask'].to(device)
                )
                batch_probs = torch.softmax(outputs.logits, dim=1)[:, 1].detach().cpu().numpy()
                batch_preds = (batch_probs > 0.5).astype(int)

                probs.extend(batch_probs)
                preds.extend(batch_preds)
                true_labels.extend(batch['label'].cpu().numpy())
                true_probs.extend(batch['prob'].cpu().numpy())

        acc = accuracy_score(true_labels, preds)
        spearman_corr, _ = spearmanr(true_probs, probs)
        acc_scores.append(acc)
        spearman_scores.append(spearman_corr)

    print(f"\n{model_name} Results:")
    print(f"Average Accuracy: {np.mean(acc_scores):.4f} ± {np.std(acc_scores):.4f}")
    print(f"Average Spearman Correlation: {np.mean(spearman_scores):.4f} ± {np.std(spearman_scores):.4f}")


models_params = {
    'bert-large-uncased': {'lr': 2.63e-5, 'batch_size': 16, 'epochs': 5}
}

for model_name, params in models_params.items():
    train_and_evaluate(model_name, params)

Using device: cuda


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/571 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.34G [00:00<?, ?B/s]

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-large-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/usr/local/lib/python3.11/dist-packages/transformers/optimization.py:591: FutureWarning: This implementation of AdamW is deprecated and will be removed in a future version. Use the PyTorch implementation torch.optim.AdamW instead, or set `no_deprecation_warning=True` to disable this warning
  warnings.warn(
bert-large-uncased Epoch 5 Seed 42: 100%|██████████| 62/62 [01:09<00:00,  1.12s/it]
Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-large-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/usr/local/lib/pyth


bert-large-uncased Results:
Average Accuracy: 0.6481 ± 0.0222
Average Spearman Correlation: 0.3653 ± 0.0253


##### Electra Large

In [ ]:
import random
import numpy as np
import torch
import gc
from torch.utils.data import DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification, AdamW
from sklearn.metrics import accuracy_score
from scipy.stats import spearmanr
from tqdm import tqdm
import pandas as pd

# Check for GPU availability
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

# Ensure Labels are Integers and Within Range
labeled_df = cleaned_df[cleaned_df['label'].notna()].copy()
labeled_df['label'] = labeled_df['label'].apply(lambda x: 1 if x == 'Hallucination' else 0).astype(int)

if 'p(Hallucination)' not in labeled_df.columns:
    raise ValueError("Missing `p(Hallucination)` field.")

labeled_df['combined_input'] = (
    labeled_df['hyp'].fillna('') + ' ' +
    labeled_df['src'].fillna('') + ' ' +
    labeled_df['tgt'].fillna('') + ' ' +
    labeled_df['ref'].fillna('') + ' ' +
    labeled_df['model'].fillna('') + ' ' +
    labeled_df['task'].fillna('')
)

train_df = labeled_df[labeled_df['Dataset'].str.contains('Val')]
test_df = labeled_df[labeled_df['Dataset'].str.contains('Test')]

X_train = train_df['combined_input'].tolist()
y_train = train_df['label'].tolist()
p_train = train_df['p(Hallucination)'].tolist()

X_test = test_df['combined_input'].tolist()
y_test = test_df['label'].tolist()
p_test = test_df['p(Hallucination)'].tolist()

class HallucinationDataset(torch.utils.data.Dataset):
    def __init__(self, texts, labels, probs, tokenizer, max_len=128):
        self.texts = texts
        self.labels = labels
        self.probs = probs
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        encoding = self.tokenizer(
            self.texts[idx], padding='max_length', truncation=True,
            max_length=self.max_len, return_tensors='pt'
        )
        return {
            'input_ids': encoding['input_ids'].squeeze(0),
            'attention_mask': encoding['attention_mask'].squeeze(0),
            'label': torch.tensor(self.labels[idx], dtype=torch.long),
            'prob': torch.tensor(self.probs[idx], dtype=torch.float)
        }

def create_datasets(model_name):
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    train_dataset = HallucinationDataset(X_train, y_train, p_train, tokenizer)
    test_dataset = HallucinationDataset(X_test, y_test, p_test, tokenizer)
    return train_dataset, test_dataset

def train_and_evaluate(model_name, params, seeds=[42, 52, 62]):
    acc_scores, spearman_scores = [], []
    for seed in seeds:
        random.seed(seed)
        np.random.seed(seed)
        torch.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

        train_dataset, test_dataset = create_datasets(model_name)

        model = AutoModelForSequenceClassification.from_pretrained(
            model_name, num_labels=2, ignore_mismatched_sizes=True
        ).to(device)

        loader = DataLoader(train_dataset, batch_size=params['batch_size'], shuffle=True, drop_last=True)
        optimizer = AdamW(model.parameters(), lr=params['lr'])

        model.train()
        for epoch in range(params['epochs']):
            total_loss = 0
            for batch in tqdm(loader, desc=f"{model_name} Epoch {epoch+1} Seed {seed}"):
                optimizer.zero_grad()
                outputs = model(
                    input_ids=batch['input_ids'].to(device),
                    attention_mask=batch['attention_mask'].to(device),
                    labels=batch['label'].to(device)
                )
                loss = outputs.loss
                loss.backward()
                optimizer.step()
                total_loss += loss.item()

            gc.collect(); torch.cuda.empty_cache()

        model.eval()
        test_loader = DataLoader(test_dataset, batch_size=params['batch_size'], shuffle=False)
        probs, preds, true_labels, true_probs = [], [], [], []

        with torch.no_grad():
            for batch in test_loader:
                outputs = model(
                    input_ids=batch['input_ids'].to(device),
                    attention_mask=batch['attention_mask'].to(device)
                )
                batch_probs = torch.softmax(outputs.logits, dim=1)[:, 1].detach().cpu().numpy()
                batch_preds = (batch_probs > 0.5).astype(int)

                probs.extend(batch_probs)
                preds.extend(batch_preds)
                true_labels.extend(batch['label'].cpu().numpy())
                true_probs.extend(batch['prob'].cpu().numpy())

        acc = accuracy_score(true_labels, preds)
        spearman_corr, _ = spearmanr(true_probs, probs)
        acc_scores.append(acc)
        spearman_scores.append(spearman_corr)

    print(f"\n{model_name} Results:")
    print(f"Average Accuracy: {np.mean(acc_scores):.4f} ± {np.std(acc_scores):.4f}")
    print(f"Average Spearman Correlation: {np.mean(spearman_scores):.4f} ± {np.std(spearman_scores):.4f}")


models_params = {
   'google/electra-large-discriminator': {'lr': 2e-5, 'batch_size': 16, 'epochs': 5}
}

for model_name, params in models_params.items():
    train_and_evaluate(model_name, params)

Using device: cuda


/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/668 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/1.34G [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.34G [00:00<?, ?B/s]

Some weights of ElectraForSequenceClassification were not initialized from the model checkpoint at google/electra-large-discriminator and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/usr/local/lib/python3.11/dist-packages/transformers/optimization.py:591: FutureWarning: This implementation of AdamW is deprecated and will be removed in a future version. Use the PyTorch implementation torch.optim.AdamW instead, or set `no_deprecation_warning=True` to disable this warning
  warnings.warn(

google/electra-large-discriminator Epoch 1 Seed 42:   0%|          | 0/62 [00:00<?, ?it/s]
google/electra-large-discriminator Epoch 1 Seed 42:   2%|▏         | 1/62 [00:02<02:07,  2.10s/it]
google/electra-large-discriminator Epoch 1 Seed 42:   3%|▎         | 2/62 [00:03<01:28,  1.48s/it]
google/electra-lar


google/electra-large-discriminator Results:
Average Accuracy: 0.6667 ± 0.0384
Average Spearman Correlation: 0.2961 ± 0.2555


##### Roberta Large

In [ ]:
import random
import numpy as np
import torch
import gc
from torch.utils.data import DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification, AdamW
from sklearn.metrics import accuracy_score
from scipy.stats import spearmanr
from tqdm import tqdm
import pandas as pd

# Check for GPU availability
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

# Ensure Labels are Integers and Within Range
labeled_df = cleaned_df[cleaned_df['label'].notna()].copy()
labeled_df['label'] = labeled_df['label'].apply(lambda x: 1 if x == 'Hallucination' else 0).astype(int)

if 'p(Hallucination)' not in labeled_df.columns:
    raise ValueError("Missing `p(Hallucination)` field.")

labeled_df['combined_input'] = (
    labeled_df['hyp'].fillna('') + ' ' +
    labeled_df['src'].fillna('') + ' ' +
    labeled_df['tgt'].fillna('') + ' ' +
    labeled_df['ref'].fillna('') + ' ' +
    labeled_df['model'].fillna('') + ' ' +
    labeled_df['task'].fillna('')
)

train_df = labeled_df[labeled_df['Dataset'].str.contains('Val')]
test_df = labeled_df[labeled_df['Dataset'].str.contains('Test')]

X_train = train_df['combined_input'].tolist()
y_train = train_df['label'].tolist()
p_train = train_df['p(Hallucination)'].tolist()

X_test = test_df['combined_input'].tolist()
y_test = test_df['label'].tolist()
p_test = test_df['p(Hallucination)'].tolist()

class HallucinationDataset(torch.utils.data.Dataset):
    def __init__(self, texts, labels, probs, tokenizer, max_len=128):
        self.texts = texts
        self.labels = labels
        self.probs = probs
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        encoding = self.tokenizer(
            self.texts[idx], padding='max_length', truncation=True,
            max_length=self.max_len, return_tensors='pt'
        )
        return {
            'input_ids': encoding['input_ids'].squeeze(0),
            'attention_mask': encoding['attention_mask'].squeeze(0),
            'label': torch.tensor(self.labels[idx], dtype=torch.long),
            'prob': torch.tensor(self.probs[idx], dtype=torch.float)
        }

def create_datasets(model_name):
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    train_dataset = HallucinationDataset(X_train, y_train, p_train, tokenizer)
    test_dataset = HallucinationDataset(X_test, y_test, p_test, tokenizer)
    return train_dataset, test_dataset

def train_and_evaluate(model_name, params, seeds=[42, 52, 62]):
    acc_scores, spearman_scores = [], []
    for seed in seeds:
        random.seed(seed)
        np.random.seed(seed)
        torch.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

        train_dataset, test_dataset = create_datasets(model_name)

        model = AutoModelForSequenceClassification.from_pretrained(
            model_name, num_labels=2, ignore_mismatched_sizes=True
        ).to(device)

        loader = DataLoader(train_dataset, batch_size=params['batch_size'], shuffle=True, drop_last=True)
        optimizer = AdamW(model.parameters(), lr=params['lr'])

        model.train()
        for epoch in range(params['epochs']):
            total_loss = 0
            for batch in tqdm(loader, desc=f"{model_name} Epoch {epoch+1} Seed {seed}"):
                optimizer.zero_grad()
                outputs = model(
                    input_ids=batch['input_ids'].to(device),
                    attention_mask=batch['attention_mask'].to(device),
                    labels=batch['label'].to(device)
                )
                loss = outputs.loss
                loss.backward()
                optimizer.step()
                total_loss += loss.item()

            gc.collect(); torch.cuda.empty_cache()

        model.eval()
        test_loader = DataLoader(test_dataset, batch_size=params['batch_size'], shuffle=False)
        probs, preds, true_labels, true_probs = [], [], [], []

        with torch.no_grad():
            for batch in test_loader:
                outputs = model(
                    input_ids=batch['input_ids'].to(device),
                    attention_mask=batch['attention_mask'].to(device)
                )
                batch_probs = torch.softmax(outputs.logits, dim=1)[:, 1].detach().cpu().numpy()
                batch_preds = (batch_probs > 0.5).astype(int)

                probs.extend(batch_probs)
                preds.extend(batch_preds)
                true_labels.extend(batch['label'].cpu().numpy())
                true_probs.extend(batch['prob'].cpu().numpy())

        acc = accuracy_score(true_labels, preds)
        spearman_corr, _ = spearmanr(true_probs, probs)
        acc_scores.append(acc)
        spearman_scores.append(spearman_corr)

    print(f"\n{model_name} Results:")
    print(f"Average Accuracy: {np.mean(acc_scores):.4f} ± {np.std(acc_scores):.4f}")
    print(f"Average Spearman Correlation: {np.mean(spearman_scores):.4f} ± {np.std(spearman_scores):.4f}")


models_params = {
    'roberta-large-mnli': {'lr': 2e-05, 'batch_size': 8, 'epochs': 3}
}

for model_name, params in models_params.items():
    train_and_evaluate(model_name, params)

Using device: cuda


tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/688 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.43G [00:00<?, ?B/s]

Some weights of the model checkpoint at roberta-large-mnli were not used when initializing RobertaForSequenceClassification: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
- This IS expected if you are initializing RobertaForSequenceClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing RobertaForSequenceClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at roberta-large-mnli and are newly initialized because the shapes did not match:
- classifier.out_proj.bias: found shape torch.Size([3]) in the checkpoint and torch.Size([2]) in the model instantiated
- classifier.out_


roberta-large-mnli Results:
Average Accuracy: 0.7448 ± 0.0076
Average Spearman Correlation: 0.5759 ± 0.0083


##### Microsoft DEBERTA Large

In [ ]:
torch.cuda.empty_cache()
import random
import numpy as np
import torch
import gc
from torch.utils.data import DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification, AdamW
from sklearn.metrics import accuracy_score
from scipy.stats import spearmanr
from tqdm import tqdm
import pandas as pd

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

labeled_df = cleaned_df[cleaned_df['label'].notna()].copy()
labeled_df['label'] = labeled_df['label'].apply(lambda x: 1 if x == 'Hallucination' else 0).astype(int)

if 'p(Hallucination)' not in labeled_df.columns:
    raise ValueError("Missing `p(Hallucination)` field.")

labeled_df['combined_input'] = (
    labeled_df['hyp'].fillna('') + ' ' +
    labeled_df['src'].fillna('') + ' ' +
    labeled_df['tgt'].fillna('') + ' ' +
    labeled_df['ref'].fillna('') + ' ' +
    labeled_df['model'].fillna('') + ' ' +
    labeled_df['task'].fillna('')
)

train_df = labeled_df[labeled_df['Dataset'].str.contains('Val')]
test_df = labeled_df[labeled_df['Dataset'].str.contains('Test')]

X_train = train_df['combined_input'].tolist()
y_train = train_df['label'].tolist()
p_train = train_df['p(Hallucination)'].tolist()

X_test = test_df['combined_input'].tolist()
y_test = test_df['label'].tolist()
p_test = test_df['p(Hallucination)'].tolist()

class HallucinationDataset(torch.utils.data.Dataset):
    def __init__(self, texts, labels, probs, tokenizer, max_len=128):
        self.texts = texts
        self.labels = labels
        self.probs = probs
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        encoding = self.tokenizer(
            self.texts[idx], padding='max_length', truncation=True,
            max_length=self.max_len, return_tensors='pt'
        )
        return {
            'input_ids': encoding['input_ids'].squeeze(0),
            'attention_mask': encoding['attention_mask'].squeeze(0),
            'label': torch.tensor(self.labels[idx], dtype=torch.long),
            'prob': torch.tensor(self.probs[idx], dtype=torch.float)
        }

def create_datasets(model_name):
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    train_dataset = HallucinationDataset(X_train, y_train, p_train, tokenizer)
    test_dataset = HallucinationDataset(X_test, y_test, p_test, tokenizer)
    return train_dataset, test_dataset

def train_and_evaluate(model_name, params, seeds=[42, 52, 62]):
    acc_scores, spearman_scores = [], []
    for seed in seeds:
        random.seed(seed)
        np.random.seed(seed)
        torch.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

        train_dataset, test_dataset = create_datasets(model_name)

        model = AutoModelForSequenceClassification.from_pretrained(
            model_name, num_labels=2, ignore_mismatched_sizes=True
        ).to(device)

        loader = DataLoader(train_dataset, batch_size=params['batch_size'], shuffle=True, drop_last=True)
        optimizer = AdamW(model.parameters(), lr=params['lr'])

        model.train()
        for epoch in range(params['epochs']):
            total_loss = 0
            for batch in tqdm(loader, desc=f"{model_name} Epoch {epoch+1} Seed {seed}"):
                optimizer.zero_grad()
                outputs = model(
                    input_ids=batch['input_ids'].to(device),
                    attention_mask=batch['attention_mask'].to(device),
                    labels=batch['label'].to(device)
                )
                loss = outputs.loss
                loss.backward()
                optimizer.step()
                total_loss += loss.item()

            gc.collect(); torch.cuda.empty_cache()

        model.eval()
        test_loader = DataLoader(test_dataset, batch_size=params['batch_size'], shuffle=False)
        probs, preds, true_labels, true_probs = [], [], [], []

        with torch.no_grad():
            for batch in test_loader:
                outputs = model(
                    input_ids=batch['input_ids'].to(device),
                    attention_mask=batch['attention_mask'].to(device)
                )
                batch_probs = torch.softmax(outputs.logits, dim=1)[:, 1].detach().cpu().numpy()
                batch_preds = (batch_probs > 0.5).astype(int)

                probs.extend(batch_probs)
                preds.extend(batch_preds)
                true_labels.extend(batch['label'].cpu().numpy())
                true_probs.extend(batch['prob'].cpu().numpy())

        acc = accuracy_score(true_labels, preds)
        spearman_corr, _ = spearmanr(true_probs, probs)
        acc_scores.append(acc)
        spearman_scores.append(spearman_corr)

    print(f"\n{model_name} Results:")
    print(f"Average Accuracy: {np.mean(acc_scores):.4f} ± {np.std(acc_scores):.4f}")
    print(f"Average Spearman Correlation: {np.mean(spearman_scores):.4f} ± {np.std(spearman_scores):.4f}")


models_params = {
   'microsoft/deberta-v3-large': {'lr': 2e-05, 'batch_size': 8, 'epochs': 5}
}

for model_name, params in models_params.items():
    train_and_evaluate(model_name, params)

Using device: cuda


/usr/local/lib/python3.11/dist-packages/transformers/convert_slow_tokenizer.py:561: UserWarning: The sentencepiece tokenizer that you are converting to a fast tokenizer uses the byte fallback option which is not implemented in the fast tokenizers. In practice this means that the fast version of the tokenizer can produce unknown tokens whereas the sentencepiece version would have converted these unknown tokens into a sequence of byte tokens matching the original piece of text.
  warnings.warn(
Some weights of DebertaV2ForSequenceClassification were not initialized from the model checkpoint at microsoft/deberta-v3-large and are newly initialized: ['classifier.bias', 'classifier.weight', 'pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/usr/local/lib/python3.11/dist-packages/transformers/optimization.py:591: FutureWarning: This implementation of AdamW is deprecated and will be removed


microsoft/deberta-v3-large Results:
Average Accuracy: 0.7425 ± 0.0013
Average Spearman Correlation: 0.5985 ± 0.0107
